# Handle multiple exceptions

When a block of code can raise several different types of exception, you need a strategy for handling each one appropriately. This guide shows you how to handle multiple exception types, re-raise exceptions, and use exception chaining to preserve diagnostic context.

**Prerequisites:**

- Basic knowledge of `try`/`except` blocks
- Familiarity with common built-in exception types (`ValueError`, `TypeError`, `KeyError`, and so on)
- Understanding of how to raise exceptions with the `raise` statement

## The problem

Real-world functions often interact with multiple subsystems, each of which can fail in different ways. For example, a function that reads configuration from a file might encounter a `FileNotFoundError`, a `PermissionError`, or a `ValueError` when parsing the contents. Handling all of these with a single, generic `except` clause loses important information about what went wrong. You need a way to respond to each exception type with the appropriate recovery action, whilst keeping your code clean and maintainable.

## Handling different exceptions with separate `except` clauses

The most straightforward approach is to write a separate `except` clause for each exception type. Python evaluates the `except` clauses in order and executes the first one that matches.

In [ ]:
def convert_to_int(value: str) -> int | None:
    """Convert a string to an integer, handling several failure modes.

    Args:
        value: The string to convert.

    Returns:
        The integer value, or None if the conversion fails.
    """
    try:
        return int(value)
    except ValueError:
        print(f"Cannot convert {value!r} to an integer: not a valid number.")
        return None
    except TypeError:
        print(f"Cannot convert {type(value).__name__} to an integer: expected a string.")
        return None

In [ ]:
# Valid input
print(convert_to_int("42"))
print()

# Triggers ValueError
print(convert_to_int("hello"))
print()

# Triggers TypeError
print(convert_to_int(None))  # type: ignore[arg-type]

Each `except` clause handles a specific exception type with a tailored response. The order matters: Python checks each clause from top to bottom and executes the first match. If you place a more general exception type before a more specific one, the specific clause will never run.

In [ ]:
import json


def parse_json_field(raw: str, field: str) -> str | None:
    """Parse a JSON string and extract a named field.

    Args:
        raw: A JSON-encoded string.
        field: The name of the field to extract.

    Returns:
        The field value as a string, or None on failure.
    """
    try:
        data = json.loads(raw)
        return str(data[field])
    except json.JSONDecodeError as exc:
        print(f"Invalid JSON: {exc.msg} at position {exc.pos}")
        return None
    except KeyError:
        print(f"Field {field!r} not found in the parsed data.")
        return None
    except TypeError:
        print("Parsed data is not a dictionary; cannot look up fields.")
        return None

In [ ]:
# Success
print(parse_json_field('{"name": "Alice"}', "name"))
print()

# JSONDecodeError: malformed JSON
print(parse_json_field("not json", "name"))
print()

# KeyError: missing field
print(parse_json_field('{"name": "Alice"}', "age"))
print()

# TypeError: JSON is a list, not a dict
print(parse_json_field('[1, 2, 3]', "name"))

This pattern works well when each exception type requires a different recovery strategy. Use it whenever you need to log different messages, return different fallback values, or take distinct corrective actions.

## Handling multiple exceptions in a single `except` clause

When two or more exception types should be handled in the same way, you can group them in a tuple within a single `except` clause. This avoids duplicating code.

In [ ]:
def safe_lookup(data: dict, key: str) -> str:
    """Look up a key in a dictionary and return its string representation.

    Args:
        data: The dictionary to search.
        key: The key to look up.

    Returns:
        The string value, or a descriptive fallback message.
    """
    try:
        return str(data[key])
    except (KeyError, TypeError) as exc:
        # KeyError: key not in dict; TypeError: data is not subscriptable
        return f"Lookup failed: {exc}"

In [ ]:
# Success
print(safe_lookup({"colour": "blue"}, "colour"))

# KeyError handled
print(safe_lookup({"colour": "blue"}, "size"))

# TypeError handled (None is not subscriptable)
print(safe_lookup(None, "colour"))  # type: ignore[arg-type]

The parentheses around the exception types are required. Without them, Python would interpret the syntax incorrectly. The `as exc` binding gives you access to whichever exception was actually raised, so you can still inspect its details.

Use this approach when the recovery logic is identical for all the grouped exception types. If you later need different behaviour for one of them, you can always split it into a separate `except` clause.

## Re-raising exceptions with bare `raise`

Sometimes you need to perform an action when an exception occurs (for example, logging it) but still allow the exception to propagate to the caller. A bare `raise` statement inside an `except` block re-raises the current exception without altering it.

In [ ]:
def process_age(raw_age: str) -> int:
    """Parse and validate an age value from a string.

    Args:
        raw_age: The age as a string.

    Returns:
        The validated age as an integer.

    Raises:
        ValueError: If the string is not a valid positive integer.
    """
    try:
        age = int(raw_age)
    except ValueError:
        print(f"LOG: Failed to parse age from {raw_age!r}")
        raise  # Re-raise the original ValueError

    if age < 0:
        raise ValueError(f"Age must not be negative, got {age}")

    return age

In [ ]:
# Valid input
print(process_age("30"))
print()

# Invalid input: the exception is logged and then re-raised
try:
    process_age("abc")
except ValueError as exc:
    print(f"Caller received: {exc}")

The bare `raise` preserves the original traceback, which is essential for debugging. Do not write `raise exc` inside an `except` block unless you have a specific reason to reset the traceback, because doing so discards the original call chain.

This pattern is particularly useful for logging, metrics collection, or partial cleanup that should happen before the exception continues to propagate.

## Exception chaining with `raise ... from ...`

When you handle one exception and raise a different one in its place, you should use the `raise ... from ...` syntax to **chain** them. This records the original exception as the **cause** of the new one, preserving the full diagnostic context.

In [ ]:
class ConfigurationError(Exception):
    """Raised when a configuration value is invalid or missing."""


def load_port(config: dict[str, str]) -> int:
    """Extract and validate the port number from a configuration dictionary.

    Args:
        config: A dictionary of configuration key-value pairs.

    Returns:
        The port number as an integer.

    Raises:
        ConfigurationError: If the port is missing or not a valid integer.
    """
    try:
        raw_port = config["port"]
    except KeyError as exc:
        raise ConfigurationError("'port' is missing from the configuration") from exc

    try:
        return int(raw_port)
    except ValueError as exc:
        raise ConfigurationError(
            f"'port' must be an integer, got {raw_port!r}"
        ) from exc

In [ ]:
# Success
print(load_port({"port": "8080"}))
print()

# Missing key: chained from KeyError
try:
    load_port({})
except ConfigurationError as exc:
    print(f"ConfigurationError: {exc}")
    print(f"  Caused by: {exc.__cause__!r}")
print()

# Invalid value: chained from ValueError
try:
    load_port({"port": "abc"})
except ConfigurationError as exc:
    print(f"ConfigurationError: {exc}")
    print(f"  Caused by: {exc.__cause__!r}")

The `from` keyword sets the `__cause__` attribute on the new exception. When Python displays the traceback, it shows both the new exception and its cause, connected by the message "The above exception was the direct cause of the following exception."

Without `from`, Python still records an **implicit** chain through the `__context__` attribute, but the traceback message reads "During handling of the above exception, another exception occurred," which is less informative. Using explicit chaining with `from` makes your intent clear.

You can also write `raise ... from None` to **suppress** the implicit chain entirely, which is useful when the original exception is an implementation detail that would only confuse the caller.

In [ ]:
def get_setting(config: dict[str, str], key: str) -> str:
    """Retrieve a required setting from a configuration dictionary.

    Args:
        config: A dictionary of configuration key-value pairs.
        key: The setting name to look up.

    Returns:
        The setting value.

    Raises:
        ConfigurationError: If the key is not present.
    """
    try:
        return config[key]
    except KeyError:
        # Suppress the KeyError; the caller only needs to know about
        # the missing configuration, not the internal lookup mechanism.
        raise ConfigurationError(f"Required setting {key!r} is missing") from None

In [ ]:
try:
    get_setting({}, "database_url")
except ConfigurationError as exc:
    print(f"ConfigurationError: {exc}")
    print(f"  __cause__: {exc.__cause__!r}")
    print(f"  __context__: {exc.__context__!r}")

Both `__cause__` and `__context__` are `None` because the chain was explicitly suppressed with `from None`.

## Using `Exception` as a fallback

You can add a final `except Exception` clause as a safety net to handle any exception type you did not anticipate. This is safer than a bare `except` (which would also handle `KeyboardInterrupt` and `SystemExit`), but it should still be used with caution.

!!! warning
    Handling `Exception` broadly can mask bugs. Always handle specific exception types first and use `Exception` only as a last resort. Log the exception so that unexpected failures are visible.

In [ ]:
def safe_divide(a: float, b: float) -> float | None:
    """Divide two numbers with comprehensive exception handling.

    Args:
        a: The dividend.
        b: The divisor.

    Returns:
        The quotient, or None if the division fails.
    """
    try:
        return a / b
    except ZeroDivisionError:
        print("Cannot divide by zero.")
        return None
    except TypeError as exc:
        print(f"Invalid operand types: {exc}")
        return None
    except Exception as exc:
        # Fallback for any other unexpected exception
        print(f"Unexpected exception: {type(exc).__name__}: {exc}")
        return None

In [ ]:
# Normal division
print(safe_divide(10, 3))
print()

# ZeroDivisionError
print(safe_divide(10, 0))
print()

# TypeError
print(safe_divide(10, "two"))  # type: ignore[arg-type]

The key principle is to order your `except` clauses **from most specific to most general**. Python matches them in order, so if you put `except Exception` first, none of the more specific clauses below it would ever run.

In production code, replace the `print()` calls with proper logging so that unexpected exceptions are recorded for investigation.

## Practical example: a robust data loader

The following example combines all of the techniques from this guide into a realistic function that reads a JSON file, validates its contents, and returns a processed result.

In [ ]:
import json
import tempfile
import os


class DataLoadError(Exception):
    """Raised when data cannot be loaded or validated."""


def load_user_record(filepath: str) -> dict[str, str | int]:
    """Load and validate a user record from a JSON file.

    The JSON file must contain a dictionary with 'name' (string) and
    'age' (integer) fields.

    Args:
        filepath: Path to the JSON file.

    Returns:
        A dictionary with 'name' and 'age' keys.

    Raises:
        DataLoadError: If the file cannot be read, parsed, or validated.
    """
    # Step 1: Read the file
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            raw_content = f.read()
    except (FileNotFoundError, PermissionError) as exc:
        # Group file-access exceptions; chain to preserve context
        raise DataLoadError(f"Cannot read file: {filepath}") from exc

    # Step 2: Parse JSON
    try:
        data = json.loads(raw_content)
    except json.JSONDecodeError as exc:
        raise DataLoadError(
            f"Invalid JSON in {filepath} at position {exc.pos}"
        ) from exc

    # Step 3: Validate structure
    if not isinstance(data, dict):
        raise DataLoadError(
            f"Expected a JSON object, got {type(data).__name__}"
        )

    try:
        name = data["name"]
        age = data["age"]
    except KeyError as exc:
        raise DataLoadError(f"Missing required field: {exc}") from exc

    # Step 4: Validate types
    if not isinstance(name, str):
        raise DataLoadError(
            f"'name' must be a string, got {type(name).__name__}"
        )
    if not isinstance(age, int) or isinstance(age, bool):
        raise DataLoadError(
            f"'age' must be an integer, got {type(age).__name__}"
        )

    return {"name": name, "age": age}

In [ ]:
# Create temporary JSON files for demonstration
temp_dir = tempfile.mkdtemp()

# Valid file
valid_path = os.path.join(temp_dir, "valid.json")
with open(valid_path, "w", encoding="utf-8") as f:
    json.dump({"name": "Alice", "age": 30}, f)

# Malformed JSON
bad_json_path = os.path.join(temp_dir, "bad.json")
with open(bad_json_path, "w", encoding="utf-8") as f:
    f.write("{name: invalid}")

# Missing required field
missing_field_path = os.path.join(temp_dir, "missing.json")
with open(missing_field_path, "w", encoding="utf-8") as f:
    json.dump({"name": "Bob"}, f)

print("Test files created.")

In [ ]:
# Test each scenario
test_cases: list[tuple[str, str]] = [
    ("Valid file", valid_path),
    ("Nonexistent file", "/nonexistent/user.json"),
    ("Malformed JSON", bad_json_path),
    ("Missing field", missing_field_path),
]

for label, path in test_cases:
    print(f"--- {label} ---")
    try:
        record = load_user_record(path)
        print(f"  Loaded: {record}")
    except DataLoadError as exc:
        print(f"  DataLoadError: {exc}")
        if exc.__cause__ is not None:
            print(f"    Caused by: {exc.__cause__!r}")
    print()

In [ ]:
# Cleanup temporary files
import shutil

shutil.rmtree(temp_dir)
print(f"Removed temporary directory: {temp_dir}")

This example demonstrates the following techniques working together:

- **Separate `except` clauses** for exceptions that require different handling (file access versus JSON parsing versus missing fields)
- **Tuple syntax** to group `FileNotFoundError` and `PermissionError` into a single clause, because both represent file-access failures
- **Exception chaining** with `raise ... from exc` to wrap low-level exceptions in a domain-specific `DataLoadError` whilst preserving the original cause
- **A custom exception class** (`DataLoadError`) that gives the caller a single, clear exception type to handle

## Summary

The following table summarises when to use each technique.

| Technique | When to use |
|---|---|
| Separate `except` clauses | Each exception type needs a different response |
| Tuple syntax `except (A, B)` | Multiple exception types share the same recovery logic |
| Bare `raise` | You need to log or record an exception but still propagate it |
| `raise ... from ...` | You are replacing one exception with a more meaningful one and want to preserve the cause |
| `raise ... from None` | You are replacing an exception and the original is an implementation detail |
| `except Exception` as fallback | You need a safety net for truly unexpected exceptions (use sparingly) |